# 12 - Desafios de Entrevista

Exercicios praticos para entrevistas tecnicas de dados.

---

## Parte 1: Pandas

### 1.1 Top N por Grupo

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
df = pd.DataFrame({
    'vendedor': rng.choice(['Ana', 'Joao', 'Maria'], 20),
    'produto': rng.choice(['A', 'B', 'C'], 20),
    'vendas': rng.integers(100, 1000, 20),
})

top2 = (df.groupby('vendedor')
    .apply(lambda g: g.nlargest(2, 'vendas'), include_groups=False)
    .reset_index(drop=True))
print(top2)

### 1.2 Deteccao de Anomalias

In [ ]:
dados = pd.DataFrame({
    'data': pd.date_range('2024-01-01', periods=100, freq='D'),
    'vendas': np.concatenate([np.random.normal(100, 10, 95), np.array([200, 210, 15, 5, 195])])
})

media = dados['vendas'].mean()
dp = dados['vendas'].std()
dados['anomalia'] = (dados['vendas'] - media).abs() > 2 * dp
print(f'Anomalias: {dados[dados["anomalia"]].shape[0]} registros')
print(dados[dados['anomalia']])

---

## Parte 2: DuckDB

### 2.1 Running Total com Window

In [ ]:
import duckdb

conn = duckdb.connect()
conn.execute("""
    CREATE TABLE vendas AS
    SELECT * FROM (VALUES
        ('2024-01-01', 'Ana', 100),
        ('2024-01-02', 'Ana', 150),
        ('2024-01-03', 'Ana', 200),
        ('2024-01-01', 'Joao', 300),
        ('2024-01-02', 'Joao', 50),
    ) AS t(data, vendedor, valor)
""")

result = conn.execute("""
    SELECT data, vendedor, valor,
           SUM(valor) OVER (PARTITION BY vendedor ORDER BY data) AS total_acumulado
    FROM vendas
    ORDER BY vendedor, data
""").fetchdf()
print(result)

### 2.2 Gap-and-Islands

In [ ]:
conn.execute("""
    CREATE TABLE sessoes AS
    SELECT * FROM (VALUES
        (1, '2024-01-01 10:00'), (2, '2024-01-01 10:05'), (3, '2024-01-01 10:10'),
        (4, '2024-01-01 11:00'), (5, '2024-01-01 11:05'),
        (6, '2024-01-01 14:00'), (7, '2024-01-01 14:01'), (8, '2024-01-01 14:02'),
    ) AS t(id, ts)
""")

result2 = conn.execute("""
    WITH com_lag AS (
        SELECT *, LAG(ts) OVER (ORDER BY ts) AS lag_ts
        FROM sessoes
    ),
    com_grupo AS (
        SELECT *,
               CASE WHEN ts::TIMESTAMP - lag_ts::TIMESTAMP >= INTERVAL '30 minutes'
                    THEN 1 ELSE 0 END AS novo_grupo
        FROM com_lag
    ),
    com_soma AS (
        SELECT *, SUM(novo_grupo) OVER (ORDER BY ts) AS grupo
        FROM com_grupo
    )
    SELECT MIN(ts) AS inicio, MAX(ts) AS fim, COUNT(*) AS eventos
    FROM com_soma
    GROUP BY grupo
    ORDER BY inicio
""").fetchdf()
print(result2)

---

## Parte 3: PySpark

### 3.1 Deduplicacao com Window

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.master('local[*]').appName('desafios').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

dados = [
    (1, 'Ana', '2024-01-01'), (2, 'Ana', '2024-01-01'),
    (3, 'Joao', '2024-01-02'), (4, 'Joao', '2024-01-03'),
]
df = spark.createDataFrame(dados, ['id', 'nome', 'data'])

w = Window.partitionBy('nome').orderBy(F.desc('id'))
df_dedup = df.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn')
print('Original:')
df.show()
print('Deduplicado:')
df_dedup.show()

### 3.2 Pivot

In [ ]:
vendas = [
    ('Ana', 'Jan', 100), ('Ana', 'Fev', 150), ('Ana', 'Mar', 200),
    ('Joao', 'Jan', 300), ('Joao', 'Fev', 50), ('Joao', 'Mar', 100),
]
df_vendas = spark.createDataFrame(vendas, ['vendedor', 'mes', 'valor'])

pivot = df_vendas.groupBy('vendedor').pivot('mes', ['Jan', 'Fev', 'Mar']).sum('valor')
pivot.show()

## Conclusao

Esses padroes (running total, deduplicacao com window, pivot) aparecem em 80% das entrevistas tecnicas.

In [ ]:
spark.stop()